## Mount the data lake gen2 containers. Check secrets

In [0]:
# Check secrets stored in key-vault and scope in databricks created 
dbutils.secrets.get(scope="warehouse_dbscope", key="client-id-secret")
dbutils.secrets.get(scope="warehouse_dbscope", key="client-secret")
dbutils.secrets.get(scope="warehouse_dbscope", key="endpoint-secret")

'[REDACTED]'

In [0]:
# Print the list of secrets. Grand role key vault reader before that
dbutils.secrets.list("warehouse_dbscope")

[SecretMetadata(key='client-id'),
 SecretMetadata(key='client-id-secret'),
 SecretMetadata(key='client-secret'),
 SecretMetadata(key='db-password'),
 SecretMetadata(key='db-username'),
 SecretMetadata(key='endpoint-secret'),
 SecretMetadata(key='wharehouse-kv-secret')]

In [0]:
# Unmount if it was already mounted.
if any(mount.mountPoint == "/mnt/silver" for mount in dbutils.fs.mounts()):
    dbutils.fs.unmount("/mnt/silver")

if any(mount.mountPoint == "/mnt/gold" for mount in dbutils.fs.mounts()):
    dbutils.fs.unmount("/mnt/gold")

if any(mount.mountPoint == "/mnt/bronze" for mount in dbutils.fs.mounts()):
    dbutils.fs.unmount("/mnt/bronze")

In [0]:
configs = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": dbutils.secrets.get(scope="warehouse_dbscope", key="client-id-secret"),
    "fs.azure.account.oauth2.client.secret": dbutils.secrets.get(scope="warehouse_dbscope", key="client-secret"),
    "fs.azure.account.oauth2.client.endpoint": dbutils.secrets.get(scope="warehouse_dbscope", key="endpoint-secret"),
}

# Mount containers 
dbutils.fs.mount(
    source="abfss://bronze@sadussbmwprod1.dfs.core.windows.net/",
    mount_point="/mnt/bronze",
    extra_configs=configs
)

dbutils.fs.mount (
    source = "abfss://silver@sadussbmwprod1.dfs.core.windows.net/",
    mount_point = "/mnt/silver",
    extra_configs = configs)

dbutils.fs.mount(
    source = "abfss://gold@sadussbmwprod1.dfs.core.windows.net/",
    mount_point = "/mnt/gold",
    extra_configs = configs)

True

In [0]:
%fs ls /mnt/bronze

path,name,size,modificationTime
dbfs:/mnt/bronze/codes_geo.parquet,codes_geo.parquet,242725171,1756134563000


In [0]:
files = dbutils.fs.ls("/mnt/bronze")
file_sizes = [(f.name, f.size / (1024 * 1024)) for f in files]
for name, size in file_sizes:
    print(f"File: {name}, Size: {size} MB")

File: codes_geo.parquet, Size: 231.48076152801514 MB


In [0]:
spark

In [0]:
dbutils.fs.ls("/mnt/bronze")

[FileInfo(path='dbfs:/mnt/bronze/codes_geo.parquet', name='codes_geo.parquet', size=242725171, modificationTime=1756134563000)]

In [0]:
code_df.show(5)

+----------+----------+----------+--------------+----------+--------+---------+---------+---------+---------+-----------+--------------------+--------+-------+
|    straat|huisnummer|huisletter|huistoevoeging|woonplaats|postcode|        x|        y|      lon|      lat|oppervlakte|      gebruiksdoelen|bouwjaar|     id|
+----------+----------+----------+--------------+----------+--------+---------+---------+---------+---------+-----------+--------------------+--------+-------+
|  Ringdijk|         6|         a|              |     Graft|  1484PC|115584.34|508166.06| 4.805959|52.559685|        140|     ["woonfunctie"]|    1998|9515946|
|  Ringdijk|         7|          |              |     Graft|  1484PC|115614.28|507651.25|4.8064613| 52.55506|         18|["overige gebruik...|    1987|9515947|
|  Ringdijk|         8|          |              |     Graft|  1484PC|115654.76| 507635.3|4.8070602| 52.55492|        232|     ["woonfunctie"]|    1902|9515948|
|Noordeinde|         1|          |      